# Notebook 03 — Baseline Models

Objective: train LR, DT, and RF baselines; generate comparison table, confusion matrices, ROC and PR curves.

**The test set is used ONCE here for reporting only — not for model selection.**

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from config import METRICS_DIR, FIGURES_DIR
from src.data_processing import load_splits
from src.feature_engineering import build_lr_pipeline, build_dt_pipeline, build_rf_pipeline
from src.train import run_baseline, cross_validate_model
from src.evaluate import (
    compute_metrics, print_metrics_table,
    plot_confusion_matrix, plot_roc_curves, plot_pr_curves,
    save_metrics_csv,
)

METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print('Setup complete.')

## 1. Load Processed Splits

In [ ]:
X_train, X_test, y_train, y_test = load_splits()
print(f'Train: {X_train.shape}  Test: {X_test.shape}')

## 2. Build Pipelines

Experiment A: original class distribution (no class_weight, no SMOTE).
Experiment B: class_weight='balanced' (handled in Notebook 04).

In [ ]:
pipe_lr = build_lr_pipeline(class_weight=None)
pipe_dt = build_dt_pipeline(class_weight=None)
pipe_rf = build_rf_pipeline(class_weight=None, n_estimators=100)

## 3. Cross-Validation on Training Data

Stratified 5-fold CV. Test set is NOT used here.

In [ ]:
print('=== Logistic Regression CV ===')
cv_lr = cross_validate_model(pipe_lr, X_train, y_train)

In [ ]:
print('=== Decision Tree CV ===')
cv_dt = cross_validate_model(pipe_dt, X_train, y_train)

In [ ]:
print('=== Random Forest CV ===')
cv_rf = cross_validate_model(pipe_rf, X_train, y_train)

## 4. Final Fit & Test Set Evaluation

Fit on full training data, evaluate on the untouched test set once.

In [ ]:
y_pred_lr, y_proba_lr = run_baseline(pipe_lr, X_train, y_train, X_test)
metrics_lr = compute_metrics(y_test, y_pred_lr, y_proba_lr, label='Logistic Regression')

In [ ]:
y_pred_dt, y_proba_dt = run_baseline(pipe_dt, X_train, y_train, X_test)
metrics_dt = compute_metrics(y_test, y_pred_dt, y_proba_dt, label='Decision Tree')

In [ ]:
y_pred_rf, y_proba_rf = run_baseline(pipe_rf, X_train, y_train, X_test)
metrics_rf = compute_metrics(y_test, y_pred_rf, y_proba_rf, label='Random Forest')

## 5. Comparison Table

In [ ]:
all_metrics = [metrics_lr, metrics_dt, metrics_rf]
comparison_df = print_metrics_table(all_metrics)
save_metrics_csv(comparison_df, 'baseline_results.csv')

## 6. Confusion Matrices

In [ ]:
plot_confusion_matrix(y_test, y_pred_lr, 'Logistic Regression')
plot_confusion_matrix(y_test, y_pred_dt, 'Decision Tree')
plot_confusion_matrix(y_test, y_pred_rf, 'Random Forest')

## 7. ROC and PR Curves

In [ ]:
roc_data = [
    {'label': 'Logistic Regression', 'y_true': y_test, 'y_proba': y_proba_lr},
    {'label': 'Decision Tree',       'y_true': y_test, 'y_proba': y_proba_dt},
    {'label': 'Random Forest',       'y_true': y_test, 'y_proba': y_proba_rf},
]
plot_roc_curves(roc_data)
plot_pr_curves(roc_data)

## 8. Baseline Observations

(Complete after running the cells above.)

- Best precision: ...
- Best recall: ...
- Best F1: ...
- Best PR-AUC: ...
- Proceed to Notebook 04 for imbalance strategies and tuning.